# Lab 1 | AI Digital Pathology
**Analyze a real medical microscopy image using machine learning.**

**Goal:** train an unsupervised model to group pixels by color, then explore different tissue-staining patterns.

**Dataset:** `skimage.data.immunohistochemistry()`, a real histology image of colonic glands with hematoxylin and DAB staining, made available through scikit-image.

**Safety:** This is an educational image-analysis demonstration. Cluster colors are **not** clinical diagnoses or validated cell types. No patient information is used.

**Instructions:** Runtime → Run all. Then change `NUMBER_OF_GROUPS` and rerun the analysis cells.

### 1 | Load and inspect the medical image

In [ ]:
# These libraries are available in a standard Google Colab Python runtime.
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, transform
from sklearn.cluster import MiniBatchKMeans

# Load a real public histology sample (not a synthetic illustration).
original = data.immunohistochemistry()
image = transform.resize(original, (256, 256), anti_aliasing=True)
print("Medical image shape:", original.shape)
plt.figure(figsize=(6, 5))
plt.imshow(original)
plt.title("Original histology image")
plt.axis("off")
plt.show()

### 2 | Train a model to group pixels by their color
**Machine learning idea:** K-means learns groups from examples without needing labeled training images.

In [ ]:
# GROUP CHALLENGE: change 4 to 2, 3, 5, or 6. Compare the results.
NUMBER_OF_GROUPS = 4

# A pixel contains three numbers: red, green, and blue.
pixels = image.reshape(-1, 3)
model = MiniBatchKMeans(n_clusters=NUMBER_OF_GROUPS, random_state=7,
                        n_init=3, batch_size=2048)
labels = model.fit_predict(pixels)

# Assign each pixel the average color of its learned group.
new_image = model.cluster_centers_[labels].reshape(image.shape)
label_map = labels.reshape(image.shape[:2])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image); axes[0].set_title("Original tissue")
axes[1].imshow(new_image); axes[1].set_title("AI color groups")
axes[2].imshow(label_map, cmap="tab10", vmin=0, vmax=9)
axes[2].set_title("Group map")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### 3 | Measure and explain the result

In [ ]:
# The percentage tells us how much of the image belongs to each color group.
percentages = np.bincount(labels, minlength=NUMBER_OF_GROUPS) / len(labels) * 100
for group, percent in enumerate(percentages):
    rgb = np.round(model.cluster_centers_[group] * 255).astype(int)
    print(f"Color group {group}: {percent:.1f}% of image | mean RGB {tuple(rgb)}")

plt.figure(figsize=(7, 3))
plt.bar([f"Group {i}" for i in range(NUMBER_OF_GROUPS)], percentages)
plt.ylabel("Image area (%)"); plt.title("How the model divided the medical image")
plt.ylim(0, 100); plt.show()

### Team mission
1. Compare `NUMBER_OF_GROUPS = 2`, `4`, and `6`. Which result best preserves visible details?
2. Identify one region the model separates well and one it separates poorly.
3. Prepare a **30-second demonstration**: original image, AI output, and one limitation.

**Debrief:** Why is color similarity not the same as diagnosing a disease?